# Melanoma concept analysis and perturbation

Analyze melanoma cell states using the saved concept definitions, visualize concept activity, evaluate concept perturbations, and assess associations with TCGA patient survival.

Configure the paths and analysis switches below, then run the cells in order. The single-cell input contains 4,092 cells and 10,000 preprocessed genes; do not log-transform it again. Outputs are saved to `Tutorial2/outputs/melanoma`.

## Prerequisites

- Install scConcept and prepare processed datasets using Tutorials 0–3.
- Generate the required concepts first, or use the deposited concept files in `Tutorial2/Concept`.
- Configure the input paths below. Processed `.mat` files use `Datasets/`; additional clinical, annotation, or plotting inputs are required by the corresponding analysis.

This tutorial starts from saved concepts. Run cells from top to bottom; preprocessing, topic training, and concept generation are not repeated. The shared input helpers are in `Tutorial2/tutorial_utils.py`.

Enabled SHAP/GSEA and CellTypist sections require `shap`, `gseapy`, and `celltypist`; survival requires `lifelines`. Use a kernel containing these analysis dependencies and the configured CellTypist model.


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.io import loadmat
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Locate Tutorial2 when launched from either Tutorial2 or Concept.
# =========================================================
# Locate the installed scConcept repository and tutorial inputs
# =========================================================
import os
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scConcept.py").is_file() and (p / "Tutorial2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the scConcept repository or one of its subdirectories.")
TUTORIAL_DIR = PROJECT_ROOT / "Tutorial2"
for directory in [PROJECT_ROOT, TUTORIAL_DIR]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial_utils import (ScConcept, SingleCellDataset, configure_inputs,
    input_path, dataset_path, load_mat_test, load_api_dataset, load_saved_concepts,
    score_saved_concepts, matlab_string, runtime_paths,
    compute_coherence_from_gene_lists, TD_eva_topics, read_pathways, interpretability)
PACKAGE_DATA_DIR = configure_inputs()
REPO_DIR = PROJECT_ROOT
CONCEPT_DIR = TUTORIAL_DIR / "Concept"

# Dataset paths can be overridden in Inputs/data_paths.local.json.
DATA_PATH = dataset_path("Melanoma")
REFERENCE_CONCEPT_PATH = CONCEPT_DIR / "GPT5_Melanoma_concepts.json"
GMT_PATH = input_path("gmt.Hs", "msigdb.v2024.1.Hs.symbols.gmt")
METADATA_PATH = input_path("melanoma.metadata", "cell_metadata_with_all_patient_labels.csv")
OUTPUT_DIR = TUTORIAL_DIR / "outputs/melanoma"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 1
SCORING_TOP_K = 100
PERTURBATION_TOP_K = 100  
TCGA_TOP_K = 10  # First 10 listed genes, selected before TCGA gene matching.
SHAP_GENESET_TOP_K = 100
PERTURBATION_FOLDS = (8.0, 32.0, 64.0)
N_CLASSIFIER_RUNS = 10
RUN_UMAP = True
RUN_SHAP = True  
RUN_COHERENCE = False  
RUN_CELLTYPIST = True  
CELLTYPIST_MODEL_PATH = Path.home() / ".celltypist/data/models/Immune_All_High.pkl"
N_RANDOM_RUNS = 100
RUN_TCGA_SURVIVAL = True  # Requires lifelines and the four local TCGA inputs below.
TCGA_EXPRESSION_PATH = input_path("melanoma.tcga_expression", "skcm_tcga/data_mrna_seq_v2_rsem.txt")
TCGA_SAMPLE_ORDER_PATH = input_path("melanoma.tcga_order", "tcga/skcm_tcga_HIGHPRE_10000.csv")
TCGA_STATUS_PATH = input_path("melanoma.tcga_status", "tcga/skcm_tcga_cell_anno.csv")
TCGA_MONTHS_PATH = input_path("melanoma.tcga_months", "tcga/skcm_tcga_cell_months.csv")

B_CONCEPT_NAME = "Mature B lymphocytes"
MALIGNANT_CONCEPT_NAME = "Melanocytic MITF program"
np.random.seed(SEED)


## 1. Load expression data

Load the cell-by-gene expression matrix (`bow_test`), cell-type labels (`label_test`), and gene names (`voc`) from the MAT file. Gene names and labels must follow the matrix column and row order, respectively.


In [ ]:
# Read the processed .mat schema used by ScConcept and SingleCellDataset.
X, vocab, test_label = load_mat_test(DATA_PATH, dtype=np.float32)
X = X.toarray() if sparse.issparse(X) else np.asarray(X)
if test_label is None:
    raise ValueError("Melanoma.mat must include label_test.")
assert (X >= 0).all()
print("Expression:", X.shape)
print(pd.Series(test_label).value_counts())


## 2. Load concepts

Load `GPT5_Melanoma_concepts.json`. Each concept contains a name and an ordered gene list.


In [ ]:
concept_path = REFERENCE_CONCEPT_PATH
concepts = load_saved_concepts(concept_path)

assert isinstance(concepts, list) and len(concepts) >= 2
assert all(isinstance(c.get("name"), str) and isinstance(c.get("genes"), list)
           and len(c["genes"]) > 0 for c in concepts)
assert len({c["name"] for c in concepts}) == len(concepts)
print("Concept source:", concept_path)
print(pd.DataFrame([{"name": c["name"], "n_genes": len(c["genes"])} for c in concepts]))


## 3. Score concepts and evaluate cell assignments

Standardize each gene across cells, then average the standardized expression of the first 100 genes per concept. Assign each cell to its highest-scoring concept. Report ARI, NMI, and AMI separately. The optional TC/TD calculation uses the first 10 genes per concept.


In [ ]:
gene_to_idx = {g: i for i, g in enumerate(vocab)}
new_concepts = [dict(c, genes=c["genes"][:SCORING_TOP_K]) for c in concepts]
perturbation_concepts = [dict(c, genes=c["genes"][:PERTURBATION_TOP_K]) for c in concepts]
def concept_indices(genes):
    return [gene_to_idx[g] for g in genes if g in gene_to_idx]

def assign_topics_by_concepts_zscore2(concepts, expression, gene_to_idx):
    """Use the package annotation rule for the saved concept gene lists."""
    genes = [g for g, _ in sorted(gene_to_idx.items(), key=lambda item: item[1])]
    scores, names, _ = score_saved_concepts(concepts, expression, genes, SCORING_TOP_K)
    return scores, names, np.asarray(names)[scores.argmax(axis=1)]

scores, topic_ids, pred_topic = assign_topics_by_concepts_zscore2(new_concepts, X, gene_to_idx)
metrics = {"ARI": adjusted_rand_score(test_label, pred_topic),
           "NMI": normalized_mutual_info_score(test_label, pred_topic),
           "AMI_legacy": adjusted_mutual_info_score(test_label, pred_topic)}
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "Fig3C_scConcept_clustering.csv", index=False)
pd.DataFrame(scores, columns=topic_ids).to_csv(OUTPUT_DIR / "cell_concept_scores.csv", index=False)
pd.DataFrame({"cell_index": np.arange(len(X)), "cell_type": test_label,
              "concept": pred_topic}).to_csv(OUTPUT_DIR / "cell_assignments.csv", index=False)
print(metrics)

# Select perturbation targets explicitly by concept name.
by_name = {c["name"]: c for c in perturbation_concepts}
if B_CONCEPT_NAME not in by_name or MALIGNANT_CONCEPT_NAME not in by_name:
    raise ValueError("Configure B_CONCEPT_NAME / MALIGNANT_CONCEPT_NAME to match the saved concept names.")
add_genes = by_name[B_CONCEPT_NAME]["genes"]
shap_concept_genes = next(c["genes"][:SHAP_GENESET_TOP_K] for c in concepts if c["name"] == MALIGNANT_CONCEPT_NAME)
remove_genes = by_name[MALIGNANT_CONCEPT_NAME]["genes"]


## 4. Visualize concept activity across cell types

Plot mean concept scores by annotated cell type and compare four T-cell-related concepts using boxplots with sampled cell-level points. Groups represent highest-scoring concept assignments within annotated T cells, rather than independently validated T-cell subtypes. Save the figures and complete score summaries.


In [ ]:
with plt.rc_context():
    from IPython.display import display
    # ============================================================
    # Melanoma: mean concept scores by annotated cell type
    #
    # Required existing variables:
    #   scores      : cells × concepts
    #   topic_ids   : concept names returned by scoring function
    #   test_label  : annotated cell types in the same cell order
    # ============================================================

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl

    from matplotlib.colors import TwoSlopeNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    # ============================================================
    # 1. Configuration
    # ============================================================
    OUT_DIR = str(OUTPUT_DIR)
    os.makedirs(OUT_DIR, exist_ok=True)

    GT_ORDER = ["B", "CAF", "Endo", "Macro", "NK", "T", "malignant"]

    SHOW_VALUES = True

    mpl.rcParams.update({
        "font.family": "Arial",
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 11,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # ============================================================
    # 2. Prepare and validate inputs
    # ============================================================
    if hasattr(scores, "detach"):
        S = scores.detach().cpu().numpy()
    else:
        S = np.asarray(scores)

    S = np.asarray(S, dtype=float)

    # Use the names returned alongside scores, avoiding stale variables
    concept_names_plot = [str(x) for x in topic_ids]

    labels = np.asarray(test_label).reshape(-1)

    # Handle scalar labels or single-element list/array labels
    def scalar_label(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            x = np.asarray(x).reshape(-1)
            if len(x) != 1:
                raise ValueError(f"Expected one label per cell, received: {x}")
            x = x[0]

        if isinstance(x, bytes):
            x = x.decode("utf-8")

        return str(x)

    labels = np.array([scalar_label(x) for x in labels])

    if S.ndim != 2:
        raise ValueError("scores must be a cells × concepts matrix.")

    if S.shape != (len(labels), len(concept_names_plot)):
        raise ValueError(
            f"Shape mismatch: scores={S.shape}, "
            f"labels={len(labels)}, concepts={len(concept_names_plot)}"
        )

    if not np.isfinite(S).all():
        raise ValueError("scores contains NaN or infinite values.")

    if len(set(concept_names_plot)) != len(concept_names_plot):
        raise ValueError("Concept names must be unique for this table.")

    observed_labels = list(pd.unique(labels))

    # Retain any additional labels instead of silently excluding them
    gt_order = [
        label for label in GT_ORDER
        if label in observed_labels
    ]
    gt_order += [
        label for label in observed_labels
        if label not in gt_order
    ]

    print("Cell types:", gt_order)
    print("Concepts:")
    for i, name in enumerate(concept_names_plot, start=1):
        print(f"{i}. {name}")

    # ============================================================
    # 3. Mean scores by annotated cell type
    # ============================================================
    score_df = pd.DataFrame(
        S,
        columns=concept_names_plot,
    )

    hm_mean = (
        score_df
        .groupby(labels, sort=False)
        .mean()
        .reindex(gt_order)
    )

    hm_mean.index.name = "Cell type"

    cell_counts = (
        pd.Series(labels)
        .value_counts()
        .reindex(gt_order)
    )

    display(hm_mean)

    hm_mean.to_csv(
        os.path.join(
            OUT_DIR,
            "Melanoma_celltype_mean_concept_scores.csv",
        ),
        encoding="utf-8-sig",
    )

    cell_counts.rename("n_cells").to_csv(
        os.path.join(
            OUT_DIR,
            "Melanoma_celltype_counts.csv",
        ),
        encoding="utf-8-sig",
    )

    # ============================================================
    # 4. Draw heatmap
    # ============================================================
    n_rows, n_cols = hm_mean.shape

    fig, ax = plt.subplots(
        figsize=(
            max(8.0, n_cols * 0.83),
            max(4.5, n_rows * 0.52),
        ),
        dpi=220,
    )

    # Symmetric scale around zero, without clipping any values
    limit = float(np.abs(hm_mean.to_numpy()).max())
    limit = max(limit, 1e-6)

    norm = TwoSlopeNorm(
        vmin=-limit,
        vcenter=0,
        vmax=limit,
    )

    cmap = plt.get_cmap("RdBu_r")

    im = ax.imshow(
        hm_mean.to_numpy(),
        cmap=cmap,
        norm=norm,
        aspect="auto",
        interpolation="nearest",
    )

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))

    ax.set_xticklabels(
        hm_mean.columns,
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_yticklabels(hm_mean.index)

    ax.set_xlabel("Concept", labelpad=10)
    ax.set_ylabel("Annotated cell type", labelpad=10)

    ax.set_title(
        "Melanoma",
        fontweight="bold",
        pad=12,
    )

    # Cell boundaries
    ax.set_xticks(
        np.arange(-0.5, n_cols, 1),
        minor=True,
    )
    ax.set_yticks(
        np.arange(-0.5, n_rows, 1),
        minor=True,
    )

    ax.grid(
        which="minor",
        color="gray",
        linewidth=1.0,
        alpha=0.20,
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    ax.tick_params(
        which="major",
        length=0,
        pad=7,
    )

    # Outer border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_color("black")

    # Numerical annotations with contrast-aware text
    if SHOW_VALUES:
        for i in range(n_rows):
            for j in range(n_cols):
                value = hm_mean.iloc[i, j]

                red, green, blue, _ = cmap(norm(value))
                luminance = (
                    0.2126 * red
                    + 0.7152 * green
                    + 0.0722 * blue
                )

                ax.text(
                    j,
                    i,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if luminance < 0.48 else "black",
                )

    # Short colorbar
    cax = inset_axes(
        ax,
        width="3.5%",
        height="48%",
        loc="center left",
        bbox_to_anchor=(1.03, 0, 1, 1),
        bbox_transform=ax.transAxes,
        borderpad=0,
    )

    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Mean concept score", fontsize=10)
    cb.ax.tick_params(labelsize=9)

    fig.subplots_adjust(
        left=0.15,
        right=0.85,
        bottom=0.40,
        top=0.90,
    )

    # ============================================================
    # 5. Save
    # ============================================================
    output_stem = os.path.join(
        OUT_DIR,
        "Melanoma_celltype_mean_concept_scores",
    )

    fig.savefig(
        output_stem + ".pdf",
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        output_stem + ".png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.show()

    print("Saved:", output_stem + ".pdf")
    print("Saved:", output_stem + ".png")
    plt.close(fig)


In [ ]:
with plt.rc_context():
    from IPython.display import display
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # ============================================================
    # 1. Inputs and configuration
    # ============================================================
    OUT_DIR = str(OUTPUT_DIR)
    os.makedirs(OUT_DIR, exist_ok=True)

    S = np.asarray(scores, dtype=float)
    names = [str(x) for x in topic_ids]
    labels = np.asarray(test_label).reshape(-1).astype(str)

    assert S.shape == (len(labels), len(names))
    assert np.isfinite(S).all()
    assert len(set(names)) == len(names)

    T_LABEL = "T"

    T_CONCEPTS = [
        "Cytotoxic T/NK response",
        "Naive/central memory T cells",
        "T follicular helper cells",
        "Regulatory T cells",
    ]

    missing = [name for name in T_CONCEPTS if name not in names]
    if missing:
        raise ValueError(
            f"Concept names not found: {missing}\n"
            f"Available names: {names}"
        )

    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 11,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # ============================================================
    # 2. Cell-level scores and highest-scoring assignments
    # ============================================================
    score_df = pd.DataFrame(S, columns=names)
    score_df.index.name = "cell_index"

    winner_idx = S.argmax(axis=1)
    winner_names = np.asarray(names)[winner_idx]

    t_mask = labels == T_LABEL

    if not t_mask.any():
        raise ValueError(
            f"No cells labeled {T_LABEL!r}. "
            f"Available labels: {np.unique(labels)}"
        )

    t_df = score_df.loc[t_mask].copy()
    t_df["assigned_concept"] = winner_names[t_mask]

    assignment_counts = (
        t_df["assigned_concept"]
        .value_counts()
        .rename_axis("assigned_concept")
        .to_frame("n_T_cells")
    )

    assignment_counts["fraction_of_T_cells"] = (
        assignment_counts["n_T_cells"] / len(t_df)
    )

    print(f"Total annotated T cells: {len(t_df)}")
    display(assignment_counts)

    # ============================================================
    # 3. Secondary scores above zero
    #    Evaluate only the four selected T-related concepts
    # ============================================================
    positive = t_df[T_CONCEPTS].gt(0)

    # Exclude the winning concept from secondary-score counts
    secondary_positive = positive.copy()

    for concept in T_CONCEPTS:
        secondary_positive.loc[
            t_df["assigned_concept"].eq(concept),
            concept,
        ] = False

    t_df["n_positive_T_related_scores"] = positive.sum(axis=1)
    t_df["n_positive_secondary_T_scores"] = (
        secondary_positive.sum(axis=1)
    )

    summary_rows = []

    for assigned, group in t_df.groupby("assigned_concept", sort=False):
        for measured in T_CONCEPTS:
            if measured == assigned:
                continue

            values = group[measured]

            summary_rows.append({
                "assigned_concept": assigned,
                "measured_secondary_concept": measured,
                "n_cells": len(group),
                "mean_score": values.mean(),
                "median_score": values.median(),
                "fraction_score_above_zero": values.gt(0).mean(),
            })

    df_secondary = pd.DataFrame(summary_rows)

    print("Secondary scores within each assignment group:")
    display(df_secondary)

    # Cells with at least two positive T-related scores,
    # including at least one score beyond the winning concept
    candidate_cells = t_df.loc[
        (t_df["n_positive_T_related_scores"] >= 2)
        & (t_df["n_positive_secondary_T_scores"] >= 1),
        [
            "assigned_concept",
            *T_CONCEPTS,
            "n_positive_T_related_scores",
            "n_positive_secondary_T_scores",
        ],
    ]

    print(
        "T cells with ≥2 positive T-related scores "
        "and ≥1 positive secondary score: "
        f"{len(candidate_cells)}/{len(t_df)} "
        f"({len(candidate_cells) / len(t_df):.1%})"
    )

    display(candidate_cells.head(20))

    # ============================================================
    # 4. Plot distributions by highest-scoring assignment
    # ============================================================
    # Include all observed assignment groups, even unexpected ones
    group_order = [
        name for name in T_CONCEPTS
        if name in t_df["assigned_concept"].values
    ]

    short_names = {
        "Cytotoxic T/NK response": "Cytotoxic T/NK",
        "Naive/central memory T cells": "Naive/central memory T",
        "T follicular helper cells": "T follicular helper",
        "Regulatory T cells": "Regulatory T",
    }

    colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
    rng = np.random.default_rng(1)

    fig, axes = plt.subplots(
        2, 2,
        figsize=(max(8, len(group_order) * 1.5), 7.5),
        sharey=True,
    )

    for ax, measured, color in zip(
        axes.flat, T_CONCEPTS, colors
    ):
        values_by_group = [
            t_df.loc[
                t_df["assigned_concept"].eq(group),
                measured,
            ].to_numpy()
            for group in group_order
        ]

        positions = np.arange(len(group_order))

        bp = ax.boxplot(
            values_by_group,
            positions=positions,
            widths=0.5,
            patch_artist=True,
            showfliers=False,
            medianprops={"color": "black", "linewidth": 1.2},
            boxprops={"edgecolor": color, "linewidth": 1.2},
            whiskerprops={"color": color},
            capprops={"color": color},
        )

        for box in bp["boxes"]:
            box.set_facecolor(color)
            box.set_alpha(0.18)

        # Sample points for display only; summaries use all cells
        for x, values in zip(positions, values_by_group):
            shown = (
                rng.choice(values, size=250, replace=False)
                if len(values) > 250 else values
            )

            ax.scatter(
                x + rng.uniform(-0.16, 0.16, len(shown)),
                shown,
                s=7,
                color=color,
                alpha=0.20,
                linewidths=0,
                rasterized=True,
            )

        ax.axhline(
            0, color="#777777",
            linestyle="--", linewidth=0.8,
        )

        ax.set_title(short_names[measured], fontsize=12)
        ax.set_xticks(positions)
        ax.set_xticklabels(
            [
                f"{short_names.get(g, g)}\n"
                f"(n={int(assignment_counts.loc[g, 'n_T_cells'])})"
                for g in group_order
            ],
            rotation=35,
            ha="right",
            fontsize=9,
        )

        ax.set_ylabel("Concept score")
       # ax.set_xlabel("Highest-scoring concept")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle("Annotated T cells", fontsize=14, y=1.01)
    fig.tight_layout()

    stem = os.path.join(
        OUT_DIR,
        "Melanoma_T_cells_secondary_concept_scores",
    )

    fig.savefig(stem + ".pdf", bbox_inches="tight")
    fig.savefig(stem + ".png", dpi=600, bbox_inches="tight")
    plt.show()

    # ============================================================
    # 5. Save complete inspection results
    # ============================================================
    t_df.to_csv(
        stem + "_all_cells.csv",
        encoding="utf-8-sig",
    )

    df_secondary.to_csv(
        stem + "_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    candidate_cells.to_csv(
        stem + "_candidate_cells.csv",
        encoding="utf-8-sig",
    )
    plt.close(fig)


In [ ]:
if RUN_COHERENCE:
    # Retain the paper's gene matching and TC/TD definitions.
    coherence = interpretability(concepts, vocab, read_pathways(GMT_PATH), top_k=10)
    (OUTPUT_DIR / "Fig3C_interpretability.json").write_text(json.dumps(
        {"TC": coherence["TC"], "TD_legacy": coherence["TD"]}, indent=2))
else:
    print("TC/TD calculation disabled.")


## 5. Visualize expression and concept embeddings

Generate expression-based UMAPs colored by cell type and concept assignment. Compute a separate UMAP from concept scores. When patient metadata are available, also visualize patient of origin. Metadata must follow the expression matrix row order.


In [ ]:
import scanpy as sc
import anndata as ad
adata = ad.AnnData(X.copy())
adata.var_names = vocab
adata.obs_names = [f"cell_{i}" for i in range(len(X))]
adata.obs["cell_type1"] = pd.Categorical(test_label)
adata.obs["cell_type2"] = pd.Categorical(np.where(test_label == "malignant", "malignant", "nonmalignant"))
adata.obs["concept"] = pd.Categorical(pred_topic)

def save_umap(data, color, name, title):
    fig, ax = plt.subplots(figsize=(13, 7))
    cell_type_palette = {
        "T": "#E74C3C", "malignant": "#8E44AD", "B": "#3498DB",
        "Macro": "#2ECC71", "Endo": "#F39C12", "CAF": "#E67E22", "NK": "#1ABC9C"
    }
    palette = cell_type_palette if color == "cell_type1" else None
    sc.pl.umap(data, color=color, palette=palette, title=title, frameon=False,
               size=20, alpha=0.9, legend_loc="right margin", ax=ax, show=False)
    ax.set_title(title, fontsize=18)
    ax.set_xlabel("UMAP1", fontsize=16)
    ax.set_ylabel("UMAP2", fontsize=16)
    if ax.legend_ is not None:
        ax.legend_.set_frame_on(False)
        for text in ax.legend_.get_texts():
            text.set_fontsize(14)
        for handle in getattr(ax.legend_, "legend_handles", getattr(ax.legend_, "legendHandles", [])):
            if hasattr(handle, "set_sizes"):
                handle.set_sizes([60])
    ax.set_position([0.06, 0.10, 0.52, 0.80])
    fig.savefig(OUTPUT_DIR / name, bbox_inches="tight")
    plt.show()
    plt.close(fig)

if RUN_UMAP:
    
    sc.tl.pca(adata)
    n_pcs = min(50, adata.obsm["X_pca"].shape[1])
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=n_pcs)
    sc.tl.umap(adata)
    np.savez(OUTPUT_DIR / "Fig3AB_expression_embedding.npz",
             umap=adata.obsm["X_umap"], pca=adata.obsm["X_pca"],
             cell_ids=adata.obs_names.to_numpy(dtype=str), cell_types=test_label,
             concepts=np.asarray(pred_topic, dtype=str))
    save_umap(adata, "cell_type1", "Fig3A_cell_types.pdf", "Annotated cell types")
    save_umap(adata, "concept", "Fig3B_concepts.pdf", "Concept assignments")
    adata_concept = ad.AnnData(scores.astype(np.float32), obs=adata.obs.copy())
    adata_concept.var_names = topic_ids
    sc.pp.neighbors(adata_concept, use_rep="X", random_state=SEED)
    sc.tl.umap(adata_concept, random_state=SEED)
    save_umap(adata_concept, "cell_type1", "SuppFig14C_concept_cell_types.pdf", "Cell types in concept space")
    
    sc.pl.umap(adata_concept, color="concept", frameon=False, size=18,
               legend_loc="right margin", title="UMAP of cell-concept scores", show=False)
    plt.savefig(OUTPUT_DIR / "SuppFig14_concept_assignments.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    plt.close()
    if METADATA_PATH.is_file():
        metadata = pd.read_csv(METADATA_PATH)
        if len(metadata) != len(X) or "cell_type" not in metadata or not np.array_equal(metadata["cell_type"].astype(str), test_label):
            raise ValueError("Metadata rows or cell-type order do not match the MAT input.")
        patient_col = next((c for c in ("patient_id_all", "patient_type") if c in metadata), None)
        if patient_col is not None:
            adata.obs["patient"] = pd.Categorical(metadata[patient_col].astype(str).to_numpy())
            save_umap(adata, "patient", "SuppFig14B_patients.pdf", "Patient of origin")
    else:
        print("Patient metadata unavailable; Supplementary Figure 14B skipped.")


## 6. Evaluate gene importance with SHAP and GSEA

Fit logistic regression to distinguish malignant from non-malignant cells. Rank genes by mean absolute SHAP values and test enrichment of the MITF concept using preranked GSEA with 1,000 permutations. Save the enrichment plot, ranked genes, and enrichment statistics. This section requires `shap` and `gseapy`.


In [ ]:
if RUN_SHAP:
    import shap
    import gseapy as gp
    y_malignant = (test_label == "malignant").astype(int)
    shap_classifier = LogisticRegression(penalty="l2", C=1.0, solver="liblinear", max_iter=1000, random_state=SEED)
    shap_classifier.fit(X, y_malignant)
    explainer = shap.LinearExplainer(shap_classifier, X, feature_perturbation="interventional")
    shap_values = np.asarray(explainer.shap_values(X))
    if shap_values.shape != X.shape:
        raise ValueError("Expected binary-classifier SHAP values with shape cells by genes.")
    df_shap = pd.DataFrame({"gene": vocab, "mean_abs_shap": np.abs(shap_values).mean(axis=0)}).sort_values("mean_abs_shap", ascending=False)
    df_shap.to_csv(OUTPUT_DIR / "Fig3D_gene_SHAP_ranking.csv", index=False)
    # The supplied SHAP notebook also tests overlap with its top 100 genes.
    # This is a SHAP validation statistic, not the CellMarker analysis in Fig. 3E.
    from scipy.stats import hypergeom
    shap_top100 = set(df_shap.head(100)["gene"])
    mitf_in_universe = set(shap_concept_genes) & set(vocab)
    overlap = mitf_in_universe & shap_top100
    overlap_stats = dict(background_genes=len(vocab), concept_genes=len(mitf_in_universe),
                         top_shap_genes=len(shap_top100), overlap=len(overlap),
                         hypergeometric_p=float(hypergeom.sf(len(overlap) - 1, len(vocab), len(mitf_in_universe), len(shap_top100))),
                         overlapping_genes=sorted(overlap))
    (OUTPUT_DIR / "SHAP_top100_overlap.json").write_text(json.dumps(overlap_stats, indent=2))
    ranked_genes = df_shap.set_index("gene")["mean_abs_shap"]
    pre_res = gp.prerank(rnk=ranked_genes, gene_sets={MALIGNANT_CONCEPT_NAME: shap_concept_genes},
                         min_size=5, max_size=500, permutation_num=1000, seed=0,
                         outdir=str(OUTPUT_DIR / "Fig3D_SHAP_enrichment"), verbose=True)
    pre_res.res2d.to_csv(OUTPUT_DIR / "Fig3D_enrichment.csv", index=False)
    plot_axes = pre_res.plot(terms=pre_res.res2d.Term.iloc[0])
    enrichment_ax = plot_axes[0] if isinstance(plot_axes, list) else plot_axes
    enrichment_ax.figure.set_size_inches(6, 5)
    enrichment_ax.figure.savefig(OUTPUT_DIR / "Fig3D_SHAP_enrichment.pdf", bbox_inches="tight")
    plt.show()
    plt.close(enrichment_ax.figure)
else:
    print("Figure 3D skipped; enable RUN_SHAP after installing its optional dependencies.")


## 7. Perturb concepts and project cell states

In malignant cells, multiply B-cell concept gene expression by 8, 32, or 64 and divide MITF concept gene expression by the same factor. Use the first 100 genes per concept. Genes shared by both concepts receive both operations. Fit PCA and UMAP to unperturbed B and malignant cells, then project perturbed cells into the same reference space.


In [ ]:
def perturb_expression(expression, add_genes, remove_genes, fold):
    perturbed = np.array(expression, copy=True)
    add_idx, remove_idx = concept_indices(add_genes), concept_indices(remove_genes)
    if not add_idx or not remove_idx:
        raise ValueError("Both perturbation gene sets must overlap the input vocabulary.")
    perturbed[:, add_idx] *= fold
    perturbed[:, remove_idx] /= fold
    return perturbed

b_mask = test_label == "B"
malignant_mask = test_label == "malignant"
assert b_mask.any() and malignant_mask.any()
print("Matched B/MITF genes:", len(concept_indices(add_genes)), len(concept_indices(remove_genes)))
if RUN_UMAP:
    from sklearn.decomposition import PCA
    import umap
    reference_X = np.vstack([X[b_mask], X[malignant_mask]])
    reference_labels = np.array(["B cell"] * b_mask.sum() + ["Malignant"] * malignant_mask.sum())
    pca = PCA(n_components=min(50, min(reference_X.shape) - 1), random_state=0)
    X_ref_pca = pca.fit_transform(reference_X)
    umap_model = umap.UMAP(n_neighbors=15, min_dist=0.5, random_state=42)
    reference_umap = umap_model.fit_transform(X_ref_pca)
    for fold in PERTURBATION_FOLDS:
        perturbed = perturb_expression(X[malignant_mask], add_genes, remove_genes, fold)
        projected = umap_model.transform(pca.transform(perturbed))
        all_umap = np.vstack([reference_umap, projected])
        all_batches = np.r_[reference_labels, np.repeat("In silico stimulated", len(projected))]
        import matplotlib.pyplot as plt
        import seaborn as sns
        import numpy as np
        import pandas as pd

        plt.style.use('default')  
        sns.set_palette("bright")  

        data_ifn = pd.DataFrame({
            'UMAP1': all_umap[all_batches=='B cell'][:,0],
            'UMAP2': all_umap[all_batches=='B cell'][:,1],
            'cell_type': 'B cell'
        })

        data_in_silico = pd.DataFrame({
            'UMAP1': all_umap[all_batches=='Malignant'][:,0],
            'UMAP2': all_umap[all_batches=='Malignant'][:,1],
            'cell_type': 'Malignant'
        })

        data_control = pd.DataFrame({
            'UMAP1': all_umap[all_batches=='In silico stimulated'][:,0],
            'UMAP2': all_umap[all_batches=='In silico stimulated'][:,1],
            'cell_type': 'In silico stimulated'
        })

        data = pd.concat([data_ifn, data_in_silico, data_control], ignore_index=True)

        fig = plt.figure(figsize=(8, 8))
        gs = fig.add_gridspec(2, 1, height_ratios=[1, 4])

        ax_scatter = fig.add_subplot(gs[1])
        sns.scatterplot(data=data, x='UMAP1', y='UMAP2', hue='cell_type', palette=['#2171A9', '#B22222', '#F2A541'],       s=15,  
                        alpha=0.8,  
                        edgecolor='none', legend=False, ax=ax_scatter)  
        ax_scatter.set_xlabel('UMAP2')
        ax_scatter.set_ylabel('UMAP1')
        ax_scatter.invert_xaxis()
                  #palette = ['#B22222',  # dark red'#2171A9',  # blue'#F2A541']  # softer orange

        ax_kde = fig.add_subplot(gs[0], sharex=ax_scatter) #['#E41A1C',  # strong red'#377EB8',  # sky blue'#4DAF4A'] 

        #             linewidth=1.5,   ax=ax_kde, legend=False)#  )
        palette = {
            'B cell': '#2171A9',
            'Malignant': '#B22222',
            'In silico stimulated': '#F2A541'
        }

        for ct in ['B cell', 'Malignant', 'In silico stimulated']:  
            subset = data[data['cell_type'] == ct]
            sns.kdeplot(
                data=subset,
                x='UMAP1',
                fill=True,
                color=palette[ct],
                common_norm=False,
                alpha=0.7,
                linewidth=1.5,
                ax=ax_kde,
                legend=False
            )

        ax_scatter.tick_params(left=False, bottom=False,labelbottom=False,labelleft=False)
        ax_scatter.set_xlabel('')
        ax_scatter.set_ylabel('')
        ax_scatter.spines['top'].set_visible(False)

        ax_kde.set_axis_off()

        plt.tight_layout()
        fig.savefig(OUTPUT_DIR / f"Fig3F_perturbation_{fold:g}x.pdf", dpi=300, bbox_inches="tight", facecolor="white")
        np.savez(OUTPUT_DIR / f"Fig3F_coordinates_{fold:g}x.npz", reference=reference_umap, projected=projected, labels=reference_labels)
        plt.show()
        plt.close(fig)


## 8. Evaluate perturbations on held-out cells

Use 10 stratified 70:30 train/test splits. Train logistic regression on unperturbed malignant and B cells, then evaluate perturbations on held-out malignant cells. Use the same split and classifier for each strength and control. Export mean B-cell probability and the fraction classified as B cells. Random controls use two distinct concepts and exclude the specified ordered pair.


In [ ]:
binary_X = np.vstack([X[malignant_mask], X[b_mask]])
binary_y = np.r_[np.zeros(malignant_mask.sum(), dtype=int), np.ones(b_mask.sum(), dtype=int)]
rows = []
for run in range(N_CLASSIFIER_RUNS):
    seed = SEED + run
    train_idx, test_idx = train_test_split(np.arange(len(binary_y)), test_size=0.3, stratify=binary_y, random_state=seed)
    clf = LogisticRegression(solver="liblinear", random_state=seed)
    clf.fit(binary_X[train_idx], binary_y[train_idx])
    heldout_idx = test_idx[binary_y[test_idx] == 0]
    heldout_X = binary_X[heldout_idx]
    rng = np.random.default_rng(seed)
    while True:
        a, r = rng.choice(len(perturbation_concepts), size=2, replace=False)
        if (perturbation_concepts[a]["name"], perturbation_concepts[r]["name"]) != (B_CONCEPT_NAME, MALIGNANT_CONCEPT_NAME):
            break
    for fold in (1.0, *PERTURBATION_FOLDS):
        for condition, add_set, remove_set in [
            ("Specified", add_genes, remove_genes),
            ("Random concepts", perturbation_concepts[a]["genes"], perturbation_concepts[r]["genes"])]:
            xp = perturb_expression(heldout_X, add_set, remove_set, fold)
            prob = clf.predict_proba(xp)[:, 1]
            rows.append(dict(run=run, split_seed=seed, fold=fold, condition=condition,
                             mean_B_probability=float(prob.mean()), fraction_predicted_B=float((prob >= 0.5).mean()),
                             n_test_malignant=len(heldout_idx), test_accuracy=float(clf.score(binary_X[test_idx], binary_y[test_idx])),
                             random_add_concept=perturbation_concepts[a]["name"], random_remove_concept=perturbation_concepts[r]["name"]))
    np.savez(OUTPUT_DIR / f"Fig3G_split_{run}.npz", train_indices=train_idx, test_indices=test_idx)
perturbation_results = pd.DataFrame(rows)
perturbation_results.to_csv(OUTPUT_DIR / "Fig3G_classifier_runs.csv", index=False)
summary = perturbation_results.groupby(["condition", "fold"])[["mean_B_probability", "fraction_predicted_B"]].agg(["mean", "std"])
summary.to_csv(OUTPUT_DIR / "Fig3G_classifier_summary.csv")
fig, ax = plt.subplots(figsize=(6, 4))
for condition, frame in perturbation_results.groupby("condition"):
    values = frame.groupby("fold")["mean_B_probability"].agg(["mean", "std"])
    ax.errorbar(values.index, values["mean"], yerr=values["std"], marker="o", capsize=3, label=condition)
ax.set(xlabel="Perturbation fold", ylabel="Predicted B-cell probability", ylim=(0, 1.05))
ax.legend()
fig.savefig(OUTPUT_DIR / "Fig3G_B_probability.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)


## 9. Validate perturbations with CellTypist

Use the local `Immune_All_High.pkl` model to obtain the `B cells` probability for unperturbed malignant cells, specified 64-fold perturbations, and 100 random-concept perturbations. After perturbation, normalize each cell to 10,000 total expression and apply `log1p` before prediction. This preparation operates on the preprocessed expression supplied above, rather than raw counts.

Export a boxplot with cell-level points, an ECDF plot, the random-control distribution, an empirical random-run P value, and a paired cell-level Wilcoxon P value. Install `celltypist` and set the local model path before running this section.


In [ ]:
def to_dense_numpy(X):
    """
    Convert sparse matrix / numpy array into dense float32 numpy array.
    """
    if sparse.issparse(X):
        X = X.toarray()

    return np.asarray(X, dtype=np.float32)

def get_gene_indices_case_insensitive(var_names, genes):
    """
    Match genes to adata.var_names in a case-insensitive manner.
    """

    gene_to_idx = {
        str(gene).strip().upper(): i
        for i, gene in enumerate(var_names)
    }

    query_genes = [
        str(gene).strip().upper()
        for gene in genes
    ]

    genes_in = [
        gene
        for gene in query_genes
        if gene in gene_to_idx
    ]

    indices = np.asarray(
        [
            gene_to_idx[gene]
            for gene in genes_in
        ],
        dtype=int
    )

    missing = sorted(
        set(query_genes) - set(genes_in)
    )

    return indices, genes_in, missing

# ============================================================
# 2. Expression perturbation
# ============================================================

def perturb_one_group(
    adata_input,
    add_genes,
    remove_genes,
    add_fold=64.0
):
    """
    Perturb the same input cells:

      add genes    *= add_fold
      remove genes /= add_fold

    Returns:
      perturbed AnnData
      gene matching information
    """

    adata_perturbed = adata_input.copy()

    add_idx, add_found, add_missing = (
        get_gene_indices_case_insensitive(
            adata_perturbed.var_names,
            add_genes
        )
    )

    remove_idx, remove_found, remove_missing = (
        get_gene_indices_case_insensitive(
            adata_perturbed.var_names,
            remove_genes
        )
    )

    X = to_dense_numpy(
        adata_perturbed.X
    ).copy()

    if add_idx.size > 0:
        X[:, add_idx] *= add_fold

    if remove_idx.size > 0:
        X[:, remove_idx] /= add_fold

    adata_perturbed.X = X

    match_info = {
        "n_add_requested": len(add_genes),
        "n_add_found": len(add_found),
        "n_add_missing": len(add_missing),
        "n_remove_requested": len(remove_genes),
        "n_remove_found": len(remove_found),
        "n_remove_missing": len(remove_missing),
    }

    return adata_perturbed, match_info

# ============================================================
# 3. CellTypist input preprocessing
# ============================================================

def prepare_for_celltypist(
    adata_input,
    already_log_normalized=False
):
    """
    Prepare AnnData for CellTypist.

    CellTypist input should generally be:
      normalize_total(target_sum=1e4)
      log1p

    Do not provide z-scored or negative-valued data.
    """

    adata_ct = adata_input.copy()

    X = adata_ct.X

    if sparse.issparse(X):
        min_value = X.min()
    else:
        min_value = np.nanmin(
            np.asarray(X)
        )

    if min_value < 0:
        raise ValueError(
            "CellTypist input contains negative values. "
            "Do not use z-scored or centered expression."
        )

    # Standardize gene-symbol capitalization
    adata_ct.var_names = pd.Index(
        [
            str(gene).strip().upper()
            for gene in adata_ct.var_names
        ]
    )

    adata_ct.var_names_make_unique()

    if not already_log_normalized:

        sc.pp.normalize_total(
            adata_ct,
            target_sum=1e4
        )

        sc.pp.log1p(
            adata_ct
        )

    return adata_ct

# ============================================================
# 4. Run CellTypist and extract only "B cells" probability
# ============================================================

def run_celltypist_b_probability(
    adata_input,
    model,
    b_cell_label="B cells",
    already_log_normalized=False
):
    """
    Run CellTypist and extract only one independent probability column:

        "B cells"

    Do not sum "B cells", "B-cell lineage", or "Plasma cells",
    because CellTypist uses independent one-vs-rest classifiers.
    """

    adata_ct = prepare_for_celltypist(
        adata_input=adata_input,
        already_log_normalized=already_log_normalized
    )

    prediction = celltypist.annotate(
        adata_ct,
        model=model,
        majority_voting=False
    )

    probability_df = (
        prediction
        .probability_matrix
        .copy()
    )

    if b_cell_label not in probability_df.columns:
        raise ValueError(
            f"CellTypist label '{b_cell_label}' was not found.\n"
            f"Available labels:\n"
            f"{probability_df.columns.tolist()}"
        )

    predicted_labels = (
        prediction
        .predicted_labels["predicted_labels"]
        .astype(str)
        .to_numpy()
    )

    b_probability = (
        probability_df[b_cell_label]
        .astype(float)
        .to_numpy()
    )

    if np.nanmin(b_probability) < 0 or np.nanmax(b_probability) > 1:
        raise ValueError(
            f"Invalid '{b_cell_label}' probability range: "
            f"{np.nanmin(b_probability)} to "
            f"{np.nanmax(b_probability)}"
        )

    hard_is_b = (
        predicted_labels == b_cell_label
    )

    return {
        "b_probability": b_probability,
        "predicted_labels": predicted_labels,
        "hard_is_b": hard_is_b,
        "prediction": prediction,
        "b_cell_label": b_cell_label,
    }

def summarize_b_result(result):
    """
    Summarize one CellTypist result.
    """

    b_probability = result["b_probability"]

    return {
        "n_cells": len(b_probability),
        "fraction_predicted_as_B": (
            result["hard_is_b"].mean()
        ),
        "mean_B_probability": (
            np.mean(b_probability)
        ),
        "median_B_probability": (
            np.median(b_probability)
        ),
        "std_B_probability": (
            np.std(
                b_probability,
                ddof=1
            )
        ),
    }

# ============================================================


In [ ]:
if RUN_CELLTYPIST:
    import celltypist
    from celltypist import models
    from scipy.stats import wilcoxon
    if not CELLTYPIST_MODEL_PATH.is_file():
        raise FileNotFoundError("Set CELLTYPIST_MODEL_PATH to a local Immune_All_High.pkl file.")
    celltypist_model = models.Model.load(str(CELLTYPIST_MODEL_PATH))
    malignant_adata = adata[malignant_mask].copy()
    no_result = run_celltypist_b_probability(malignant_adata, celltypist_model)
    specified_adata, _ = perturb_one_group(malignant_adata, add_genes, remove_genes, 64.0)
    specified_result = run_celltypist_b_probability(specified_adata, celltypist_model)
    no_probability = no_result["b_probability"]
    specified_probability = specified_result["b_probability"]
    rng = np.random.default_rng(SEED)
    random_probabilities, random_rows = [], []
    for run in range(N_RANDOM_RUNS):
        while True:
            a, r = rng.choice(len(perturbation_concepts), 2, replace=False)
            if (perturbation_concepts[a]["name"], perturbation_concepts[r]["name"]) != (B_CONCEPT_NAME, MALIGNANT_CONCEPT_NAME):
                break
        perturbed, _ = perturb_one_group(malignant_adata, perturbation_concepts[a]["genes"], perturbation_concepts[r]["genes"], 64.0)
        result = run_celltypist_b_probability(perturbed, celltypist_model)
        random_probabilities.append(result["b_probability"])
        random_rows.append(dict(run=run, add_concept=perturbation_concepts[a]["name"], remove_concept=perturbation_concepts[r]["name"]))
    random_probabilities = np.asarray(random_probabilities)
    random_mean = random_probabilities.mean(axis=0)
    specified_delta = float((specified_probability - no_probability).mean())
    random_delta = (random_probabilities - no_probability).mean(axis=1)
    empirical_p = (1 + int((random_delta >= specified_delta).sum())) / (N_RANDOM_RUNS + 1)
    differences = specified_probability - random_mean
    paired_p = float(wilcoxon(differences, alternative="greater").pvalue) if np.any(differences) else 1.0
    (OUTPUT_DIR / "SuppFig15_statistics.json").write_text(json.dumps({"empirical_random_run_p": empirical_p,
        "paired_cell_wilcoxon_p": paired_p, "specified_mean_increase": specified_delta}, indent=2))
    pd.DataFrame(random_rows).to_csv(OUTPUT_DIR / "SuppFig15_random_concepts.csv", index=False)
    pd.DataFrame({"cell_index": np.flatnonzero(malignant_mask), "unperturbed": no_probability,
        "specified": specified_probability, "random_mean": random_mean}).to_csv(OUTPUT_DIR / "SuppFig15_cell_probabilities.csv", index=False)
    np.save(OUTPUT_DIR / "SuppFig15_random_probabilities.npy", random_probabilities)
    import os
    out_dir = str(OUTPUT_DIR)
    random_mean_probability_per_cell = random_mean
    random_mean_probabilities = random_probabilities.mean(axis=1)
    specified_mean_probability = specified_probability.mean()
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    out_dir = str(OUTPUT_DIR)
    os.makedirs(out_dir, exist_ok=True)

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 12,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.linewidth": 1,
        "pdf.fonttype": 42,
    })

    data = [
        no_probability,
        random_mean_probability_per_cell,
        specified_probability
    ]

    labels = ["No\nperturbation",
              "Random\nperturbation",
              "Specified\nperturbation"]

    colors = ["#7F7F7F", "#8172B2", "#C44E52"]

    fig, ax = plt.subplots(figsize=(4.5, 3.5))

    # Boxplot
    bp = ax.boxplot(
        data, positions=[1, 2, 3], widths=0.42,
        patch_artist=True, showfliers=False,
        medianprops=dict(color="black", lw=1.5),
        whiskerprops=dict(color="#555555", lw=1),
        capprops=dict(color="#555555", lw=1)
    )

    for box, c in zip(bp["boxes"], colors):
        box.set_facecolor(c)
        box.set_edgecolor(c)
        box.set_alpha(0.35)

    # Jittered points
    rng = np.random.default_rng(1)

    for i, (y, c) in enumerate(zip(data, colors), 1):
        idx = rng.choice(len(y), min(200, len(y)), replace=False)
        yy = np.asarray(y)[idx]
        xx = i + rng.uniform(-0.16, 0.16, len(yy))

        ax.scatter(
            xx, yy,
            s=9, color=c,
            alpha=0.35,
            edgecolors="none",
            zorder=3
        )

    ax.set_xticks([1, 2, 3], labels)
    ax.set_ylabel("CellTypist B-cell probability")
    ax.set_ylim(0, 1.02)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=4, width=1)

    plt.tight_layout()

    plt.savefig(
        os.path.join(out_dir, "CellTypist_perturbation_scatter_box.pdf"),
        bbox_inches="tight"
    )
    plt.savefig(
        os.path.join(out_dir, "CellTypist_perturbation_scatter_box.png"),
        dpi=600, bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    out_dir = str(OUTPUT_DIR)
    os.makedirs(out_dir, exist_ok=True)

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.linewidth": 1.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })

    colors = {
        "no": "#777777",
        "random": "#8172B2",
        "specified": "#C44E52"
    }

    fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.1))

    # ============================================================
    # Left: Cell-level ECDF
    # ============================================================
    for values, label, color in [
        (no_probability,
         "No perturbation",
         colors["no"]),

        (random_mean_probability_per_cell,
         "Random perturbation",
         colors["random"]),

        (specified_probability,
         "Specified perturbation",
         colors["specified"])
    ]:
        x = np.sort(values)
        y = np.arange(1, len(x) + 1) / len(x)

        ax[0].plot(
            x, y,
            lw=2,
            color=color,
            label=label
        )

    ax[0].set_xlabel("CellTypist B-cell probability")
    ax[0].set_ylabel("Cumulative fraction")
    ax[0].set_xlim(0, 1.1)
    ax[0].set_ylim(0, 1.02)

    ax[0].legend(
        frameon=False,
        fontsize=9,
        loc="lower right"
    )

    # ============================================================
    # Right: Random perturbation null distribution
    # ============================================================
    ax[1].hist(
        random_mean_probabilities,
        bins=15,
        color="#BDBDBD",
        edgecolor="white",
        linewidth=0.8
    )

    # No perturbation
    ax[1].axvline(
        no_probability.mean(),
        color=colors["no"],
        linestyle="--",
        linewidth=1.8,
        label="No perturbation"
    )

    # Specified perturbation
    ax[1].axvline(
        specified_mean_probability,
        color=colors["specified"],
        linewidth=2.2,
        label="Specified perturbation"
    )

    ax[1].set_ylabel("Random perturbations")

    # Legend: upper right
    ax[1].legend(
        frameon=False,
        fontsize=9,
        loc="upper right"
    )

    # Empirical P-value: lower right
    ax[1].text(
        0.97, 0.08,
        f"Empirical $P$ = {empirical_p:.4f}",
        transform=ax[1].transAxes,
        ha="right",
        va="bottom",
        fontsize=10.5
    )

    # ============================================================
    # Style
    # ============================================================
    for a in ax:
        a.spines["top"].set_visible(False)
        a.spines["right"].set_visible(False)
        a.tick_params(
            direction="out",
            length=4,
            width=1
        )

    plt.subplots_adjust(
        left=0.11,
        right=0.98,
        bottom=0.20,
        top=0.96,
        wspace=0.40
    )
    ax[1].set_xlabel("Mean CellTypist B-cell probability")
    # ============================================================
    # Save
    # ============================================================
    plt.savefig(
        os.path.join(
            out_dir,
            "CellTypist_orthogonal_validation.pdf"
        ),
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)
else:
    print("CellTypist skipped. Enable RUN_CELLTYPIST and supply the local model to run it.")


## 10. Analyze TCGA patient survival

Read the TCGA expression matrix without an additional log transformation. Use the sample-order CSV to align expression with the status and follow-up arrays; these arrays contain no independent sample identifiers and must retain their matching order.

Select the first 10 listed genes per concept before expression matching; missing genes are skipped without replacement. Standardize genes across the aligned TCGA cohort and average matched genes for each concept. Use the MITF concept as the malignant score and the mean of all other concept scores as the non-malignant score.

Exclude samples with invalid survival fields or follow-up times <= 0. Define low and high groups using the bottom and top quartiles, exclude the middle half, and generate Kaplan–Meier curves with log-rank tests. Save sample eligibility, matched genes, concept scores, group assignments, and survival results.


In [ ]:
def decode_tcga_status(value):
    """Decode explicit observed TCGA OS labels; do not use substring matching."""
    text = str(value).strip().upper()
    if text in {"1:DECEASED", "DECEASED", "DEAD", "1", "1.0"}:
        return 1.0
    if text in {"0:LIVING", "LIVING", "ALIVE", "0", "0.0"}:
        return 0.0
    return np.nan

def prepare_tcga_survival_inputs(expression_path, sample_order_path, status_path, months_path, concepts, malignant_name):
    """Prepare top-10 concept scores and aligned survival data."""
    paths = [expression_path, sample_order_path, status_path, months_path]
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f"Missing TCGA input: {path}")
    sample_ids = pd.read_csv(sample_order_path, usecols=[0]).iloc[:, 0].astype(str).tolist()
    if len(set(sample_ids)) != len(sample_ids):
        raise ValueError("TCGA sample-order IDs must be unique.")
    status_frame = pd.read_csv(status_path, header=None)
    months_frame = pd.read_csv(months_path, header=None)
    if status_frame.shape[0] != 1 or months_frame.shape[0] != 1:
        raise ValueError("Expected one-row clinical arrays.")
    statuses = status_frame.iloc[0].to_numpy()
    months = pd.to_numeric(months_frame.iloc[0], errors="coerce").to_numpy(dtype=float)
    if len(statuses) != len(sample_ids) or len(months) != len(sample_ids):
        raise ValueError("TCGA clinical array lengths do not match the sample-order CSV.")
    raw = pd.read_csv(expression_path, sep="\t").dropna(subset=["Hugo_Symbol"])
    raw["Hugo_Symbol"] = raw["Hugo_Symbol"].astype(str)
    
    # Keeping the last row reproduces that behavior before transposition.
    raw = raw.drop_duplicates("Hugo_Symbol", keep="last").set_index("Hugo_Symbol")
    missing_samples = sorted(set(sample_ids) - set(raw.columns))
    if missing_samples:
        raise ValueError(f"TCGA expression samples are missing: {missing_samples[:10]}")
    top10 = [dict(c, genes=c["genes"][:TCGA_TOP_K]) for c in concepts]
    names = [c["name"] for c in top10]
    if malignant_name not in names or len(names) < 2:
        raise ValueError("Expected a malignant concept and at least one non-malignant concept.")
    used_genes = list(dict.fromkeys(g for c in top10 for g in c["genes"] if g in raw.index))
    expr = raw.loc[used_genes, sample_ids].T.apply(pd.to_numeric, errors="raise")
    if not np.isfinite(expr.to_numpy()).all():
        raise ValueError("Non-finite values in the TCGA concept-gene expression matrix.")
    tcga_scores, tcga_names, _ = assign_topics_by_concepts_zscore2(
        top10, expr.to_numpy(), {g: i for i, g in enumerate(expr.columns)})
    score_df = pd.DataFrame(tcga_scores, index=pd.Index(sample_ids, name="sampleId"), columns=tcga_names)
    clinical = pd.DataFrame({"OS_STATUS_original": statuses,
                             "OS_STATUS": [decode_tcga_status(v) for v in statuses],
                             "OS_MONTHS": months}, index=score_df.index)
    clinical["eligible"] = (np.isfinite(clinical["OS_STATUS"]) & np.isfinite(clinical["OS_MONTHS"]) & (clinical["OS_MONTHS"] > 0))
    malignant_idx = tcga_names.index(malignant_name)
    other_idx = [i for i in range(len(tcga_names)) if i != malignant_idx]
    programs = pd.DataFrame({"malignant": tcga_scores[:, malignant_idx],
                             "non_malignant": tcga_scores[:, other_idx].mean(axis=1)}, index=score_df.index)
    matches = [{"concept": c["name"], "requested_genes": c["genes"],
                "n_requested": len(c["genes"]),
                "n_matched": sum(g in expr.columns for g in c["genes"]),
                "matched_genes": [g for g in c["genes"] if g in expr.columns],
                "missing_genes": [g for g in c["genes"] if g not in expr.columns]} for c in top10]
    return score_df, clinical, programs, matches

def quartile_groups(values):
    """Select <=Q25 and >=Q75 groups with distinct cutoffs."""
    q25, q75 = values.quantile([0.25, 0.75])
    if not np.isfinite([q25, q75]).all() or q25 >= q75:
        raise ValueError("Quartile cutoffs are not distinct; low and high groups would overlap.")
    groups = pd.Series("middle", index=values.index)
    groups.loc[values <= q25] = "low"
    groups.loc[values >= q75] = "high"
    return groups, float(q25), float(q75)

if RUN_TCGA_SURVIVAL:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    tcga_scores, tcga_clinical, tcga_programs, tcga_gene_matches = prepare_tcga_survival_inputs(
        TCGA_EXPRESSION_PATH, TCGA_SAMPLE_ORDER_PATH, TCGA_STATUS_PATH, TCGA_MONTHS_PATH,
        concepts, MALIGNANT_CONCEPT_NAME)
    print(pd.DataFrame(tcga_gene_matches)[["concept", "n_requested", "n_matched"]].to_string(index=False))
    tcga_scores.to_csv(OUTPUT_DIR / "TCGA_all_concept_scores.csv")
    tcga_clinical.to_csv(OUTPUT_DIR / "TCGA_survival_eligibility.csv")
    (OUTPUT_DIR / "TCGA_concept_gene_matches.json").write_text(json.dumps(tcga_gene_matches, indent=2))
    survival_df = tcga_programs.join(tcga_clinical).loc[tcga_clinical["eligible"]].copy()
    survival_rows = []
    for program, panel, title in [("non_malignant", "Fig3H", "Non-malignant concepts"),
                                   ("malignant", "Fig3I", MALIGNANT_CONCEPT_NAME)]:
        groups, q25, q75 = quartile_groups(survival_df[program])
        survival_df[f"{program}_group"] = groups
        low = survival_df.loc[groups == "low"]
        high = survival_df.loc[groups == "high"]
        if min(len(low), len(high)) < 5:
            raise ValueError(f"Too few samples for {panel}: high={len(high)}, low={len(low)}")
        result = logrank_test(high.OS_MONTHS, low.OS_MONTHS,
                              event_observed_A=high.OS_STATUS, event_observed_B=low.OS_STATUS)
        fig, ax = plt.subplots(figsize=(8, 6), facecolor="white")
        for group, frame, color in [("High", high, "red"), ("Low", low, "blue")]:
            km = KaplanMeierFitter()
            km.fit(frame.OS_MONTHS, frame.OS_STATUS, label=f"{title} {group.lower()} expression")
            km.plot_survival_function(ax=ax, ci_show=False, color=color, linewidth=2)
            km.survival_function_.to_csv(OUTPUT_DIR / f"{panel}_{group.lower()}_KM.csv")
            km.event_table.to_csv(OUTPUT_DIR / f"{panel}_{group.lower()}_event_table.csv")
        ax.set_title(f"{title} survival analysis", fontsize=14, fontweight="bold")
        ax.set_xlabel("Time (months)", fontsize=12)
        ax.set_ylabel("Survival probability", fontsize=12)
        ax.text(0.65, 0.75, f"p = {result.p_value:.4f}", transform=ax.transAxes, fontsize=12,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        ax.text(0.65, 0.65, f"High expression: n={len(high)}", transform=ax.transAxes, fontsize=10, color="red")
        ax.text(0.65, 0.60, f"Low expression: n={len(low)}", transform=ax.transAxes, fontsize=10, color="blue")
        ax.grid(False)
        ax.legend(fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
        fig.savefig(OUTPUT_DIR / f"{panel}_TCGA_survival.pdf", bbox_inches="tight")
        plt.show()
        plt.close(fig)
        survival_rows.append(dict(panel=panel, program=program, n_eligible=len(survival_df),
                                  n_high=len(high), n_low=len(low), q25=q25, q75=q75,
                                  logrank_statistic=float(result.test_statistic), logrank_p=float(result.p_value)))
    survival_df.to_csv(OUTPUT_DIR / "TCGA_survival_scores_and_groups.csv")
    survival_results = pd.DataFrame(survival_rows)
    survival_results.to_csv(OUTPUT_DIR / "Fig3HI_logrank_results.csv", index=False)
    print(survival_results.to_string(index=False))
else:
    print("TCGA survival skipped; set RUN_TCGA_SURVIVAL=True after configuring its inputs.")


## Output guide

| Analysis | Main output |
|---|---|
| Cell-type and concept UMAPs | `Fig3A_cell_types.pdf`, `Fig3B_concepts.pdf` |
| Cell-type mean concept scores | `Melanoma_celltype_mean_concept_scores.pdf` |
| T-cell concept distributions | `Melanoma_T_cells_secondary_concept_scores.pdf` |
| SHAP/GSEA | `Fig3D_SHAP_enrichment.pdf` |
| Perturbation projections | `Fig3F_perturbation_<fold>x.pdf` |
| Held-out classification | `Fig3G_B_probability.pdf` |
| CellTypist validation | `CellTypist_perturbation_scatter_box.pdf`, `CellTypist_orthogonal_validation.pdf` |
| TCGA survival | `Fig3H_TCGA_survival.pdf`, `Fig3I_TCGA_survival.pdf` |

The final cell saves the run configuration and package versions.


In [ ]:
import platform
import importlib.metadata as metadata
packages = ["numpy", "pandas", "scipy", "scikit-learn", "scanpy", "anndata", "umap-learn", "matplotlib", "shap", "gseapy", "celltypist", "lifelines"]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not installed"
provenance = dict(concept_mode="saved", concept_path=str(concept_path),
                  concept_sha256=hashlib.sha256(concept_path.read_bytes()).hexdigest(),
                  data_path=str(DATA_PATH), seed=SEED, classifier_split_seeds=list(range(SEED, SEED + N_CLASSIFIER_RUNS)),
                  perturbation_top_k=PERTURBATION_TOP_K, scoring_top_k=SCORING_TOP_K, shap_geneset_top_k=SHAP_GENESET_TOP_K, tcga_top_k=TCGA_TOP_K,
                  folds=list(PERTURBATION_FOLDS), python=platform.python_version(), packages=versions,
                  enabled=dict(umap=RUN_UMAP, shap=RUN_SHAP, coherence=RUN_COHERENCE, celltypist=RUN_CELLTYPIST, tcga_survival=RUN_TCGA_SURVIVAL),
                  tcga_inputs={"expression": str(TCGA_EXPRESSION_PATH), "sample_order": str(TCGA_SAMPLE_ORDER_PATH),
                               "status": str(TCGA_STATUS_PATH), "months": str(TCGA_MONTHS_PATH)})
provenance["package_paths"] = runtime_paths()
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(provenance, indent=2))
print("Outputs:", OUTPUT_DIR)
